# Lecture 4 — Post-Training, Hands On

### AI Tooling Seminar · `huggingface/trl`

This notebook is the running half of the Lecture 4 assignment. The reading half lives in
`trl_assignment.md` — do Part 1 there first; several exercises below assume you've already
found the relevant code by hand.

---

## Kaggle setup (do this before running anything)

In the right-hand sidebar:

| Setting | Value |
|---|---|
| **Accelerator** | `GPU T4 x2` or `GPU P100` |
| **Internet** | `On` — required for `pip install` and Hub downloads |
| **Persistence** | Optional; nothing here needs it |

Everything uses **Qwen2.5-0.5B**, small enough that all five training runs fit
comfortably in 16 GB at full precision. Each run is capped at a few dozen steps —
the goal is to watch the mechanics, not to produce a good model.

> ⚠️ **Honesty note from whoever built this.** The exercises were validated by reading
> TRL's source directly — every constructor signature, config field and default below was
> checked against the actual code, and every dataset was confirmed to exist on the Hub.
> But they were **not executed on a GPU** before being handed to you. If a cell fails,
> that's worth reporting back rather than assuming you did something wrong. Check
> `trl.__version__` in Cell 1 against the version noted in the lecture first — the API
> moves, and that's a lesson in itself.

## 0 · Install and check the environment

Note the version that prints. The lecture was built against **1.10.0.dev0**. If yours differs,
expect some of what follows to have shifted — and treat that as data, not as breakage.

In [ ]:
!pip install -q -U trl transformers datasets accelerate peft

import os, torch, trl, transformers

# Pin to a single GPU: Kaggle's T4 x2 otherwise triggers multi-GPU paths
# that complicate the batch-size divisibility rules in Exercise E.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

print('trl          ', trl.__version__)
print('transformers ', transformers.__version__)
print('torch        ', torch.__version__)
print('GPU          ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — fix the Accelerator setting')
print('VRAM (GB)    ', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else '-')

---
## Exercise A · Check the lecture against the installed library

Slide 18 claimed six stable trainers and roughly twenty-five experimental ones, with PPO
on the experimental side. Don't take that on faith — the version you just installed may
not be the version the lecture was written against.

**Do:** run the cell. **Notice:** whether the counts and the location of PPO still match.

In [ ]:
import trl, inspect, pkgutil
from pathlib import Path

stable = sorted(n for n in dir(trl) if n.endswith('Trainer'))
print('Stable trainers exported by trl:')
for n in stable: print('  ', n)

exp_dir = Path(inspect.getfile(trl)).parent / 'experimental'
exp = sorted(m.name for m in pkgutil.iter_modules([str(exp_dir)]))
print(f'\ntrl.experimental contains {len(exp)} modules:')
print('  ', ', '.join(exp))

print('\nIs PPO stable?     ', 'PPOTrainer' in stable)
print('Is PPO experimental?', 'ppo' in exp)

**Reflect:** if the numbers moved, what moved — did something get promoted out of
`experimental`, or was something new added to it? What would you have concluded if you'd
trusted a blog post from a year ago instead of running this?

---
## Exercise B · SFT — learning from a demonstration

The simplest signal on slide 3: someone wrote the answer, copy it. Watch the loss.

**Do:** run it. **Notice:** the dataset column, and that you never specified a loss anywhere.

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'

sft_ds = load_dataset('trl-lib/Capybara', split='train[:400]')
print('columns:', sft_ds.column_names)
print('one record:', {k: str(v)[:120] for k, v in sft_ds[0].items()})

sft_trainer = SFTTrainer(
    model=MODEL,
    train_dataset=sft_ds,
    args=SFTConfig(
        output_dir='/kaggle/working/sft',
        max_steps=30,
        per_device_train_batch_size=2,
        max_length=512,
        learning_rate=2e-5,
        logging_steps=5,
        report_to='none',
    ),
)
sft_trainer.train()

**Reflect:** the dataset has a `messages` column and nothing else — no scores, no
comparisons, no labels. Which of the three signals from slide 3 is that, and what is the
ceiling on how good this model can get from this data alone?

---
## Exercise C · Reward modelling — learning a judge

Now the second signal. Same base network, but the language-modelling head is replaced by
a single scalar. Slide 15 showed the loss; here you watch it run.

**Do:** run it. **Notice:** the `accuracy` and `margin` values in the log output — they
come from the metrics block you read on slide 15.

In [ ]:
from trl import RewardTrainer, RewardConfig

pref_ds = load_dataset('trl-lib/ultrafeedback_binarized', split='train[:400]')
print('columns:', pref_ds.column_names)

rm_trainer = RewardTrainer(
    model=MODEL,
    train_dataset=pref_ds,
    args=RewardConfig(
        output_dir='/kaggle/working/rm',
        max_steps=30,
        per_device_train_batch_size=2,
        max_length=512,
        logging_steps=5,
        report_to='none',
    ),
)
rm_trainer.train()

**Reflect:** `accuracy` here is the fraction of pairs where the chosen response outscored
the rejected one. What accuracy would an untrained scalar head get, and why? And why is
`mean_reward` not a number you can compare against anyone else's reward model?

---
## Exercise D · DPO — the same judgment, no judge

Identical data to Exercise C. Completely different mechanism: no reward model is trained,
and the frozen reference does the anchoring directly.

**Do:** run it. **Notice:** you passed the *same dataset* as Exercise C, and got a
generator out instead of a scorer.

In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_trainer = DPOTrainer(
    model=MODEL,
    train_dataset=pref_ds,
    args=DPOConfig(
        output_dir='/kaggle/working/dpo',
        max_steps=30,
        per_device_train_batch_size=2,
        max_length=512,
        beta=0.1,
        logging_steps=5,
        report_to='none',
    ),
)
dpo_trainer.train()

**Reflect:** Exercises C and D consumed the same records and produced different kinds of
artifact. Slide 13's table said RewardTrainer wants *implicit* prompt and DPOTrainer wants
*explicit* — look at `pref_ds.column_names` again. Which is this, and what did TRL do about it?

Also: watch memory. DPO holds a reference model that Exercise C did not. Did you notice?

---
## Exercise E · GRPO — a verifier, and no data at all

The third signal. Note what the dataset is: prompts, nothing else. The reward is the
four-line function from slide 16, which you can read in full in `trl/rewards/format_rewards.py`.

**Two constraints worth understanding rather than copying:**

- `num_generations` defaults to **8**, and the effective batch size
  (`per_device_train_batch_size × gradient_accumulation_steps × num_processes`) must be
  evenly divisible by it. We use 4 and 4. (Strictly, TRL computes the generation batch as
  `steps_per_generation × per_device_train_batch_size`, and `steps_per_generation` falls back
  to `gradient_accumulation_steps` when unset — so here it's 1 × 4 = 4, giving one prompt with
  four completions. That is exactly one GRPO *group*: four samples of the same prompt, scored
  against their own mean.)
- `use_vllm` defaults to **False**, so generation runs through ordinary `model.generate()`.
  Slower, but it means no vLLM install and no server. This is why the cell is the slowest here.

**Do:** run it. **Notice:** the `reward` column in the logs, and whether it climbs.

In [ ]:
from trl import GRPOTrainer, GRPOConfig
from trl.rewards import think_format_reward
from datasets import Dataset

# Prompt-only data — we build it by hand to make the shape unmistakable.
questions = [
    'What is 17 + 25?', 'Name a prime number between 10 and 20.',
    'What is the capital of Portugal?', 'What is 8 times 7?',
    'How many days are in a leap year?', 'What is the square root of 144?',
    'What is 100 divided by 4?', 'Which planet is closest to the Sun?',
]
SYS = 'Think step by step inside <think> and </think> tags, then give your answer.'
grpo_ds = Dataset.from_dict({'prompt': [
    [{'role': 'system', 'content': SYS}, {'role': 'user', 'content': q}]
    for q in questions * 8
]})
print('columns:', grpo_ds.column_names, '| rows:', len(grpo_ds))

grpo_trainer = GRPOTrainer(
    model=MODEL,
    reward_funcs=think_format_reward,
    train_dataset=grpo_ds,
    args=GRPOConfig(
        output_dir='/kaggle/working/grpo',
        max_steps=20,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=1,
        num_generations=4,
        max_completion_length=96,
        logging_steps=1,
        report_to='none',
    ),
)
grpo_trainer.train()

**Reflect:** no preference data was collected, no reward model was trained, and no human
judged anything. Where did the training signal come from, mechanically? And `think_format_reward`
returns 1.0 for correctly-placed tags regardless of whether the answer is right — what is
this run actually teaching the model, and what is it emphatically *not* teaching it?

---
## Exercise F · The anchor that wasn't there

Slide 8 presented the KL anchor as the standard defence against optimizing into nonsense.
Now go read the default:

```python
from trl import GRPOConfig
GRPOConfig.__dataclass_fields__['beta'].default
```

**Do:** run the cell below, which prints that default and then reruns GRPO with the anchor
switched on. **Notice:** what appears in the logs the second time that wasn't there the first.

In [ ]:
from trl import GRPOConfig, DPOConfig

print('GRPOConfig beta default:', GRPOConfig.__dataclass_fields__['beta'].default)
print('DPOConfig  beta default:', DPOConfig.__dataclass_fields__['beta'].default)
print()
print(GRPOConfig.__dataclass_fields__['beta'].metadata['help'][:220])

anchored = GRPOTrainer(
    model=MODEL,
    reward_funcs=think_format_reward,
    train_dataset=grpo_ds,
    args=GRPOConfig(
        output_dir='/kaggle/working/grpo_kl',
        max_steps=10,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=1,
        num_generations=4,
        max_completion_length=96,
        beta=0.04,          # anchor ON
        logging_steps=1,
        report_to='none',
    ),
)
anchored.train()

**Reflect:** this is the sharpest finding in the whole lesson, and the lecture didn't tell
you about it — you had to read the default.

`DPOConfig.beta` defaults to `0.1`: the anchor is on. `GRPOConfig.beta` defaults to `0.0`:
the reference model isn't even loaded. The help text says so explicitly and cites the
DeepSeek-R1 work as the reason.

So: why would a preference method keep the anchor while a verifier-driven RL method drops
it? Think about what each one is optimizing against, and which of the two has a scorer that
can be exploited. Then ask whether the argument fully holds — is `think_format_reward`
really unexploitable?

---
## Overall reflection

You have now run four of the six stable trainers. Three of them consumed data you were
handed; one generated its own. Two produced a chat model, one produced a scorer, one
produced a model that reasons in tags whether or not it reasons correctly.

**Write a few sentences:** if a team handed you 50,000 records and asked which of these
methods to use, what would you need to look at first — and what would you refuse to decide
before seeing it?

---

### If something broke

Distinguish two cases before debugging, because the fix differs:

- **Never worked.** The exercise was wrong when written. Report it.
- **Worked, then stopped.** An upstream dependency moved — a config field renamed, a default
  changed, a dataset re-uploaded with different columns. Check `trl.__version__` against the
  lecture's `1.10.0.dev0` and read `MIGRATION.md` in the repo.

The second case is not a flaw in the assignment; it is the subject of slide 18.